**ФИО студента:** [Иванов Иван Иванович]

**Задание:** [выберите между `inpainting` / `outpainting`]

**Группа:** [номер группы]

**Дата сдачи:** [дата]

## Блок настройки окружения

Код для импорта библиотек, для формирования глобальных констант, других конфигураций (если требуется). Постарайтесь здесь зафиксировать все seed, которые сможете

In [ ]:
import os
import random

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import hashlib
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from plotly.subplots import make_subplots
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"


## Блок "Данные"

Входной файл проверяется по SHA-256, форме и точному расположению NaN. Загрузка через `mmap_mode="r"` не создаёт лишнюю копию куба, поскольку исходный dtype уже `float32`.

Пять соседних срезов дают модели 2.5D-контекст по глубине: центральный срез восстанавливается с учётом двух предыдущих и двух следующих. Патч 320×160 охватывает локальную геологическую структуру и ограничивает расход памяти. Синтетическая маска 213×103 соответствует размеру реальной скрытой области, а её положение и отражения патчей варьируются только для train.

Разбиение выполнено по индексам центральных срезов, а не по отдельным пикселям. Между train, validation и test оставлены защитные интервалы, поэтому их пятисрезовые контексты не пересекаются и данные не протекают между выборками.


In [ ]:
DATA_PATH = Path("input_seismic_data/inpainting_cube_missed_data.npy")
EXPECTED_SHAPE = (580, 951, 462)
EXPECTED_SHA256 = "1aa956cfb12a75536ddd815f5d6f21abfbcccf474aec6de0d0a04c76b6eab1e5"

sha256 = hashlib.sha256()
with DATA_PATH.open("rb") as file:
    for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()
assert actual_sha256 == EXPECTED_SHA256, (
    f"SHA-256 не совпадает: ожидался {EXPECTED_SHA256}, получен {actual_sha256}"
)

cube = np.load(DATA_PATH, mmap_mode="r")
assert cube.shape == EXPECTED_SHAPE
assert cube.dtype == np.float32
print(cube.shape)

real_mask = np.zeros(cube.shape, dtype=bool)
real_mask[:, 368:581, 179:282] = True

for slice_index in range(cube.shape[0]):
    assert np.array_equal(np.isnan(cube[slice_index]), real_mask[slice_index]), (
        f"Расположение NaN не совпадает с заявленным в срезе {slice_index}"
    )

missing_count = int(real_mask.sum())
print(
    f"Проверка NaN пройдена: {missing_count:,} пропущенных значений, "
    f"{100 * missing_count / cube.size:.2f}% куба"
)

PATCH_H = 320
PATCH_W = 160

MASK_H = 213
MASK_W = 103
REGIONS = (
    (0, 368, 0, 462),
    (581, 951, 0, 462),
    (0, 951, 0, 179),
    (0, 951, 282, 462)
)


In [ ]:
sample = cube[::4, ::4, ::4]
sample_known = np.isfinite(sample) & ~real_mask[::4, ::4, ::4]
mean = float(sample[sample_known].mean())
std = max(float(sample[sample_known].std()), 1e-6)


In [ ]:
def make_samples(centers, samples_per_center, augment, seed):
    rng = np.random.default_rng(seed)
    samples = []

    for center in centers:
        for _ in range(samples_per_center):
            y_min, y_max, z_min, z_max = REGIONS[rng.integers(len(REGIONS))]
            y = int(rng.integers(y_min, y_max - PATCH_H + 1))
            z = int(rng.integers(z_min, z_max - PATCH_W + 1))
            mask_y = int(rng.integers(16, PATCH_H - MASK_H - 15))
            mask_z = int(rng.integers(16, PATCH_W - MASK_W - 15))
            flip_h = augment and bool(rng.integers(2))
            flip_v = augment and bool(rng.integers(2))
            samples.append((center, y, z, mask_y, mask_z, flip_h, flip_v))

    return samples


In [ ]:
class InpaintingDataset(Dataset):
    def __init__(self, cube, samples, mean, std):
        self.cube = cube
        self.samples = samples
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        center, y, z, mask_y, mask_z, flip_h, flip_v = self.samples[index]
        patch = self.cube[
            center - 2:center + 3,
            y:y + PATCH_H,
            z:z + PATCH_W
        ].copy()
        target = patch[2].copy()

        patch = (patch - self.mean) / self.std
        target = (target - self.mean) / self.std

        mask = np.zeros((PATCH_H, PATCH_W), dtype=np.float32)
        mask[mask_y:mask_y + MASK_H, mask_z:mask_z + MASK_W] = 1
        patch *= (1 - mask)[None]

        if flip_h:
            patch = patch[:, :, ::-1]
            target = target[:, ::-1]
            mask = mask[:, ::-1]

        if flip_v:
            patch = patch[:, ::-1, :]
            target = target[::-1, :]
            mask = mask[::-1, :]

        inputs = np.concatenate((patch, mask[None]), axis=0)

        return (
            torch.from_numpy(inputs.copy()).float(),
            torch.from_numpy(target[None].copy()).float(),
            torch.from_numpy(mask[None].copy()).float()
        )


In [ ]:
train_centers = np.arange(2, 402)
val_centers = np.arange(406, 492)
test_centers = np.arange(496, 578)
train_samples = make_samples(train_centers, 8, True, 42)
val_samples = make_samples(val_centers, 4, False, 43)
test_samples = make_samples(test_centers, 4, False, 44)

train_dataset = InpaintingDataset(cube, train_samples, mean, std)
val_dataset = InpaintingDataset(cube, val_samples, mean, std)
test_dataset = InpaintingDataset(cube, test_samples, mean, std)


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)
val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


## Блок "Модель"

Используется полносвёрточная 2.5D U-Net: пять соседних срезов и маска пропусков подаются как шесть каналов. `AvgPool2d` сохраняет усреднённую структуру подписанных амплитуд вместо выбора только экстремумов. Для восстановления разрешения применяются nearest-интерполяция и свёртка 3×3, что исключает шахматочные артефакты транспонированной свёртки. GroupNorm стабилизирует обучение при небольшом batch size.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU()
        )

    def forward(self, x):
        return self.block(x)


In [ ]:
class UNet25D(nn.Module):
    def __init__(self, in_channels=6, out_channels=1, base_channels=32):
        super().__init__()
        self.pool = nn.AvgPool2d(2)
        self.encoders = nn.ModuleList([
            ConvBlock(in_channels, base_channels),
            ConvBlock(base_channels, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 8)
        ])
        self.bottleneck = ConvBlock(base_channels * 8, base_channels * 16)
        self.upconvs = nn.ModuleList([
            nn.Conv2d(base_channels * 16, base_channels * 8, 3, padding=1),
            nn.Conv2d(base_channels * 8, base_channels * 4, 3, padding=1),
            nn.Conv2d(base_channels * 4, base_channels * 2, 3, padding=1),
            nn.Conv2d(base_channels * 2, base_channels, 3, padding=1)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock(base_channels * 16, base_channels * 8),
            ConvBlock(base_channels * 8, base_channels * 4),
            ConvBlock(base_channels * 4, base_channels * 2),
            ConvBlock(base_channels * 2, base_channels)
        ])
        self.head = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        height, width = x.shape[-2:]
        pad_height = (-height) % 16
        pad_width = (-width) % 16
        x = F.pad(x, (0, pad_width, 0, pad_height))

        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for upconv, decoder, skip in zip(
            self.upconvs,
            self.decoders,
            reversed(skips)
        ):
            x = F.interpolate(x, scale_factor=2, mode="nearest")
            x = upconv(x)
            x = torch.cat((x, skip), dim=1)
            x = decoder(x)

        x = self.head(x)
        return x[..., :height, :width]


model = UNet25D()
print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")


## Блок "Обучение модели"

Функция потерь объединяет маскированный MSE и дифференцируемый член `0.1 · (1 − R)`, поэтому оптимизация напрямую учитывает целевую метрику корреляции. Лучшая модель выбирается по validation R. История train и validation сохраняется в TensorBoard и выводится на статичных графиках loss и R.

Задан лимит в 100 эпох и ранняя остановка после 8 эпох без улучшения. Для дообучения лучшего чекпоинта нужно установить `RESUME = True`: восстановятся веса, оптимизатор, scheduler, scaler, номер эпохи и история.


In [ ]:
def masked_mse(prediction, target, mask):
    error = (prediction - target) ** 2
    return (error * mask).sum() / mask.sum().clamp_min(1)


def masked_r(prediction, target, mask):
    prediction = prediction[mask.bool()].float()
    target = target[mask.bool()].float()
    prediction = prediction - prediction.mean()
    target = target - target.mean()
    denominator = prediction.norm() * target.norm()
    return (prediction * target).sum() / denominator.clamp_min(1e-8)


def reconstruction_loss(prediction, target, mask):
    mse_loss = masked_mse(prediction, target, mask)
    r_loss = 1 - masked_r(prediction, target, mask)
    return mse_loss + R_LOSS_WEIGHT * r_loss


In [ ]:
%load_ext tensorboard
from torch.utils.tensorboard import SummaryWriter

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

EPOCHS = 100
EARLY_STOPPING_PATIENCE = 8
R_LOSS_WEIGHT = 0.1
RESUME = False
CHECKPOINT_PATH = "best_inpainting_unet.pt"
LOG_DIR = "runs/inpainting"
writer = SummaryWriter(LOG_DIR)
best_val_r = -float("inf")
start_epoch = 1
epochs_without_improvement = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "train_r": [],
    "val_r": []
}

if RESUME:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_r = checkpoint["best_val_r"]
    history = checkpoint["history"]


In [ ]:
def train_epoch():
    model.train()
    total_loss = 0
    total_r = 0

    for inputs, target, mask in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            prediction = model(inputs)
            loss = reconstruction_loss(prediction, target, mask)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * inputs.size(0)
        total_r += masked_r(prediction.detach(), target, mask).item()

    return total_loss / len(train_loader.dataset), total_r / len(train_loader)


In [ ]:
def validation_epoch():
    model.eval()
    total_loss = 0
    total_r = 0

    with torch.inference_mode():
        for inputs, target, mask in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, enabled=use_amp):
                prediction = model(inputs)
                loss = reconstruction_loss(prediction, target, mask)

            total_loss += loss.item() * inputs.size(0)
            total_r += masked_r(prediction, target, mask).item()

    return total_loss / len(val_loader.dataset), total_r / len(val_loader)


In [ ]:
for epoch in range(start_epoch, EPOCHS + 1):
    train_loss, train_r = train_epoch()
    val_loss, val_r = validation_epoch()
    scheduler.step(val_r)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_r"].append(train_r)
    history["val_r"].append(val_r)

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/validation", val_loss, epoch)
    writer.add_scalar("R/train", train_r, epoch)
    writer.add_scalar("R/validation", val_r, epoch)
    writer.add_scalar("Learning rate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    if val_r > best_val_r:
        best_val_r = val_r
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "mean": mean,
            "std": std,
            "epoch": epoch,
            "best_val_r": best_val_r,
            "history": history
        }, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss: {train_loss:.6f} | val loss: {val_loss:.6f} | "
        f"train R: {train_r:.4f} | val R: {val_r:.4f}"
    )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Ранняя остановка на эпохе {epoch}")
        break

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
best_epoch = checkpoint["epoch"]
writer.close()


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, history["train_loss"], label="Train")
axes[0].plot(epochs, history["val_loss"], label="Validation")
axes[0].set(title="Функция потерь", xlabel="Эпоха", ylabel="Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["train_r"], label="Train")
axes[1].plot(epochs, history["val_r"], label="Validation")
axes[1].set(title="Коэффициент корреляции", xlabel="Эпоха", ylabel="R")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig("inpainting_training_history.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
%tensorboard --logdir runs/inpainting


## Блок "Тестирование модели"

Код и информация о том, как выглядят результаты предсказания на тестовой выборке

In [ ]:
model.eval()
test_loss = 0
prediction_values = []
target_values = []

with torch.inference_mode():
    for inputs, target, mask in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            prediction = model(inputs)
            loss = reconstruction_loss(prediction, target, mask)

        test_loss += loss.item() * inputs.size(0)
        prediction_values.append(prediction[mask.bool()].float().cpu())
        target_values.append(target[mask.bool()].float().cpu())

test_loss /= len(test_loader.dataset)
test_predictions = torch.cat(prediction_values)
test_targets = torch.cat(target_values)
test_r = masked_r(
    test_predictions,
    test_targets,
    torch.ones_like(test_targets)
).item()

best_train_r = history["train_r"][best_epoch - 1]
best_val_r = history["val_r"][best_epoch - 1]
print(f"Test loss: {test_loss:.6f} | Test R: {test_r:.4f}")
print(
    f"Итоговые R | train: {best_train_r:.4f} | "
    f"val: {best_val_r:.4f} | test: {test_r:.4f}"
)


In [ ]:
inputs, target, mask = next(iter(test_loader))
inputs = inputs.to(device)
target = target.to(device)
mask = mask.to(device)

with torch.inference_mode():
    with torch.autocast(device_type=device.type, enabled=use_amp):
        prediction = model(inputs)

for index in range(min(3, inputs.size(0))):
    original = target[index, 0].cpu().numpy() * std + mean
    predicted = prediction[index, 0].float().cpu().numpy() * std + mean
    sample_mask = mask[index, 0].cpu().numpy().astype(bool)

    restored = original.copy()
    restored[sample_mask] = predicted[sample_mask]

    error = np.full_like(original, np.nan)
    error[sample_mask] = np.abs(predicted[sample_mask] - original[sample_mask])

    a_min, a_max = np.percentile(original, [1, 99])

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("Оригинал", "Восстановленный срез", "Абсолютная ошибка")
    )
    fig.add_trace(
        go.Heatmap(z=original, zmin=a_min, zmax=a_max, colorscale="Gray", showscale=False),
        row=1,
        col=1
    )
    fig.add_trace(
        go.Heatmap(z=restored, zmin=a_min, zmax=a_max, colorscale="Gray", showscale=False),
        row=1,
        col=2
    )
    fig.add_trace(
        go.Heatmap(z=error, colorscale="Hot", showscale=True),
        row=1,
        col=3
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        width=1200,
        height=450,
        title_text=f"Тестовый пример {index + 1}"
    )
    fig.show()


## Блок "Инференса модели"

Сеть полносвёрточная и использует локальные признаки, поэтому после обучения на патчах 320×160 применяется ко всему срезу 951×462. Вход дополняется до размеров, кратных 16, а результат обрезается до исходной формы.

На срезах 0, 1 и двух последних индексах несуществующие контекстные кадры заполняются нулями после нормализации. Это соответствует кодированию пропусков и не дублирует крайние кадры, чего не было в обучении.


In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


def restore_slice(index):
    indices = np.arange(index - 2, index + 3)
    valid = (indices >= 0) & (indices < cube.shape[0])
    frames = np.zeros((5, *cube.shape[1:]), dtype=np.float32)
    frames[valid] = cube[indices[valid]]
    frames = (frames - mean) / std
    frames[~valid] = 0
    frames = np.nan_to_num(frames, nan=0.0, posinf=0.0, neginf=0.0)

    sample_mask = real_mask[index].astype(np.float32)
    frames *= (1 - sample_mask)[None]

    inputs = np.concatenate((frames, sample_mask[None]), axis=0)
    inputs = torch.from_numpy(inputs[None].copy()).float().to(device)

    with torch.inference_mode():
        with torch.autocast(device_type=device.type, enabled=use_amp):
            prediction = model(inputs)

    prediction = prediction[0, 0].float().cpu().numpy() * std + mean
    restored = cube[index].copy()
    restored[real_mask[index]] = prediction[real_mask[index]]
    return restored


In [ ]:
def show_restored_slice(index):
    restored = restore_slice(index)
    original = cube[index].copy()
    missing = real_mask[index]

    original_display = original.copy()
    original_display[missing] = np.nan

    restored_region = np.full_like(restored, np.nan)
    restored_region[missing] = restored[missing]

    known = original[~missing]
    known = known[np.isfinite(known)]
    a_min, a_max = np.percentile(known, [1, 99])

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=("Исходный срез", "Восстановленная область", "Результат")
    )
    fig.add_trace(
        go.Heatmap(z=original_display, zmin=a_min, zmax=a_max, colorscale="Gray", showscale=False),
        row=1,
        col=1
    )
    fig.add_trace(
        go.Heatmap(z=restored_region, zmin=a_min, zmax=a_max, colorscale="Gray", showscale=False),
        row=1,
        col=2
    )
    fig.add_trace(
        go.Heatmap(z=restored, zmin=a_min, zmax=a_max, colorscale="Gray", showscale=False),
        row=1,
        col=3
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(width=1200, height=550, title_text=f"Срез {index}")
    fig.show()


In [ ]:
slice_slider = widgets.IntSlider(
    min=0,
    max=cube.shape[0] - 1,
    step=1,
    value=cube.shape[0] // 2,
    description="Срез",
    continuous_update=False
)

widgets.interact(show_restored_slice, index=slice_slider)


## Блок сохранения артефактов

Код и информация о том, как сохранялась модель и восстановленный куб (shape финального куба должен совпадать с shape'ом входного куба, который был скачан по ссылке)

In [ ]:
restored_cube = cube.copy()

for index in range(cube.shape[0]):
    restored_cube[index] = restore_slice(index)

assert restored_cube.shape == cube.shape

output_path = "inpainting_cube_restored.npy"
model_output_path = "inpainting_unet_inference.pt"
np.save(output_path, restored_cube)
torch.save(model.state_dict(), model_output_path)
print(f"Восстановленный куб сохранен: {output_path}")
print(f"Модель для инференса сохранена: {model_output_path}")
